In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

import joblib
import os

In [2]:
df = pd.read_csv(
    "data/processed/final_selected_dataset.csv"
)

print(df.shape)

df.head()

(250000, 21)


,Make_and_Model,Vehicle_Type,Year_of_Manufacture,Road_Conditions,Weather_Conditions,Route_Info,Usage_Hours,Load_Capacity,Actual_Load,Engine_Temperature,...,Battery_Status,Oil_Quality,Vibration_Levels,Tire_Pressure,Failure_History,Anomalies_Detected,Diagnostic_Trouble_Code_Count,CAN_Message_Rate_Hz,Sensor_Packet_Loss_Rate,Maintenance_Required
0,1,4,2016.263210,4,4,1,12156.999890,14134.916000,8856.297726,88.187864,...,90.838404,82.617827,0.688063,39.766979,0.000000,0.000000,0.000000,42.834041,0.001632,0
1,0,2,2016.101678,3,5,1,14921.423040,13161.594650,7915.782384,90.827148,...,98.972817,89.485681,0.858262,37.269779,0.102551,1.696143,0.561199,42.828402,0.008061,0
2,3,3,2011.648228,0,4,1,19723.877880,9844.764975,5884.328371,88.566975,...,80.778262,91.016870,1.344360,37.255419,1.318466,0.000000,0.000000,29.492107,0.008887,0
3,4,0,2020.812445,1,5,0,5175.324145,3499.397353,494.375580,99.434739,...,87.467459,86.231550,2.041878,38.598132,0.162311,0.000000,0.184538,44.711910,0.006513,0
4,4,1,2022.314726,2,4,2,2120.124439,6078.939605,1380.182251,93.293504,...,80.282667,102.884309,1.450554,37.227710,1.060079,0.000000,0.897322,50.780901,0.008521,0


In [3]:
X = df.drop(
    "Maintenance_Required",
    axis=1
)

y = df["Maintenance_Required"]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(200000, 20)
(50000, 20)


In [5]:
train_df = pd.concat(
    [X_train, y_train],
    axis=1
)

test_df = pd.concat(
    [X_test, y_test],
    axis=1
)

os.makedirs("data/processed", exist_ok=True)

train_df.to_csv(
    "data/processed/train_dataset.csv",
    index=False
)

test_df.to_csv(
    "data/processed/test_dataset.csv",
    index=False
)

print("Train and Test datasets saved.")

Train and Test datasets saved.


In [6]:
corr = df.corr(numeric_only=True)["Maintenance_Required"]

corr = corr.sort_values(
    ascending=False,
    key=lambda x: abs(x)
)

print(corr)

Maintenance_Required             1.000000
Battery_Status                  -0.584152
Oil_Quality                     -0.579472
Fuel_Consumption                 0.575331
Vibration_Levels                 0.559709
Engine_Temperature               0.545675
CAN_Message_Rate_Hz              0.516599
Sensor_Packet_Loss_Rate          0.509179
Tire_Pressure                   -0.496414
Anomalies_Detected               0.481607
Diagnostic_Trouble_Code_Count    0.467034
Failure_History                  0.438540
Actual_Load                      0.160794
Usage_Hours                     -0.003444
Year_of_Manufacture              0.003311
Make_and_Model                  -0.001929
Road_Conditions                  0.001651
Weather_Conditions               0.001623
Vehicle_Type                    -0.000741
Load_Capacity                    0.000681
Route_Info                       0.000337
Name: Maintenance_Required, dtype: float64


In [7]:
for col in X.columns:
    print(f"{col:35} {df[col].nunique()}")

Make_and_Model                      8
Vehicle_Type                        5
Year_of_Manufacture                 248011
Road_Conditions                     5
Weather_Conditions                  6
Route_Info                          5
Usage_Hours                         247139
Load_Capacity                       244374
Actual_Load                         242339
Engine_Temperature                  249990
Fuel_Consumption                    249996
Battery_Status                      249991
Oil_Quality                         249990
Vibration_Levels                    246313
Tire_Pressure                       249982
Failure_History                     165755
Anomalies_Detected                  164277
Diagnostic_Trouble_Code_Count       173007
CAN_Message_Rate_Hz                 249997
Sensor_Packet_Loss_Rate             230507


In [9]:
models = {

    "Logistic Regression":
    LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Decision Tree":
    DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest":
    RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost":
    XGBClassifier(
        random_state=42,
        eval_metric="logloss"
    )

}

In [ ]:
results = []

for name, model in models.items():

    print("="*60)
    print(name)

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:,1]
    else:
        y_prob = y_pred

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)

    print(classification_report(y_test, y_pred))

    results.append([

        name,
        accuracy,
        precision,
        recall,
        f1,
        roc

    ])

Logistic Regression


e:\Predictive-vehicle-maintenance-with-road-analysis\Model\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


              precision    recall  f1-score   support

           0       0.87      0.94      0.91     32625
           1       0.88      0.74      0.80     17375

    accuracy                           0.87     50000
   macro avg       0.88      0.84      0.86     50000
weighted avg       0.87      0.87      0.87     50000

Decision Tree
              precision    recall  f1-score   support

           0       0.83      0.82      0.82     32625
           1       0.67      0.68      0.67     17375

    accuracy                           0.77     50000
   macro avg       0.75      0.75      0.75     50000
weighted avg       0.77      0.77      0.77     50000

Random Forest


In [ ]:
results_df = pd.DataFrame(

    results,

    columns=[

        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC-AUC"

    ]

)

results_df = results_df.sort_values(
    by="Accuracy",
    ascending=False
)

results_df

In [ ]:
results_df.set_index("Model")[

    [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ]

].plot(
    kind="bar",
    figsize=(10,6)
)

plt.title("Model Comparison (Without SMOTE)")

plt.ylabel("Score")

plt.xticks(rotation=0)

plt.grid(True)

plt.show()

In [ ]:
for name, model in models.items():

    y_pred = model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm
    )

    disp.plot()

    plt.title(name)

    plt.show()

In [ ]:
best_model_name = results_df.iloc[0]["Model"]

best_model = models[best_model_name]

os.makedirs(
    "models",
    exist_ok=True
)

joblib.dump(

    best_model,

    "models/best_model_without_smote.joblib"

)

print("Best Model :", best_model_name)